# De-excitation blip checks

Compares the nominal blip variables to their `_deexadded` twins (simulated MARLEY
nuclear de-excitation photon blips injected during `create_df`, see
`src/deex_blip_injection.py`), after the WC generic neutrino selection.
The nominal-vs-`_deexadded` comparison is the no-de-excitation vs
GENIE+INCL+MARLEY systematic bracket.

Also checks the injection bookkeeping: match rates, match-level fallbacks, and
donor-usage uniformity (each MARLEY library event and each iso1g blip donor
should be sampled approximately uniformly).

No systematics are applied here (the weights/detvar dataframes for the current
production have not been processed yet); statistical uncertainties only.


In [ ]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt

import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.file_locations import intermediate_files_location
from src.plot_helpers import make_histogram_plot
from src.df_helpers import lazy_height

# File Loading

In [ ]:
print("loading all_df.parquet...")
all_df = pl.scan_parquet(f"{intermediate_files_location}/all_df.parquet")
print(f"num events in all_df: {lazy_height(all_df)}")

deex_cols = [c for c in all_df.collect_schema().names() if c.endswith("_deexadded")]
print(f"num _deexadded columns: {len(deex_cols)}")

# Injection bookkeeping

Per-filetype summary of the de-excitation injection. Expectations (from the
GENIE+INCL+MARLEY sample): ~72% of matched overlay events have >= 1 photon and
the mean is ~1.26 photons/event; match level 0 (exact CC/NC x mode x
protons x neutrons key) should dominate, and level -2 (eligible but unmatched)
should never occur. Data / EXT / the 1-gamma overlays are skipped by design
(level -1).

In [ ]:
bk = (
    all_df.select(["filetype", "deex_marley_donor_index", "deex_marley_match_level",
                   "deex_n_photons", "deex_n_photons_injected", "deex_n_blips_injected"])
    .collect()
)
assert (bk["deex_marley_match_level"] != -2).all(), "eligible-but-unmatched events found!"

summary = (
    bk.group_by("filetype")
    .agg(
        pl.len().alias("events"),
        (pl.col("deex_marley_donor_index") >= 0).mean().alias("frac_matched"),
        (pl.col("deex_n_photons") > 0).mean().alias("frac_ge1_photon"),
        pl.col("deex_n_photons").mean().alias("mean_photons"),
        pl.col("deex_n_blips_injected").mean().alias("mean_blips_injected"),
    )
    .sort("events", descending=True)
)
with pl.Config(tbl_rows=30, float_precision=3):
    print(summary)

matched = bk.filter(pl.col("deex_marley_donor_index") >= 0)
print("\nmatch levels among matched events (0 = exact key, 1-3 = fallbacks):")
print(matched["deex_marley_match_level"].value_counts().sort("deex_marley_match_level"))

# Donor-usage uniformity

Left: how many times each MARLEY library event was sampled (among sampled
donors). Right: how many times each iso1g blip donor was used. Both should look
roughly Poisson around their mean -- a long tail of heavily reused donors would
mean some matching bin is too sparse.

In [ ]:
iso_all = (
    all_df.select("deex_iso1g_donor_indices")
    .filter(pl.col("deex_iso1g_donor_indices") != "")
    .collect()["deex_iso1g_donor_indices"]
    .str.split(",").explode().cast(pl.Int64)
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (usage, label) in zip(axes, [
    (matched["deex_marley_donor_index"].value_counts()["count"].to_numpy(),
     "MARLEY library event"),
    (iso_all.value_counts()["count"].to_numpy(), "iso1g blip donor"),
]):
    mean_use = usage.mean()
    median_use = np.median(usage)
    bins = np.arange(-0.5, max(usage.max() + 1.5, 10), 1)
    ax.hist(usage, bins=bins, histtype="step", color="#0072B2")
    ax.axvline(mean_use, color="#D55E00", linestyle=":",
               label=f"mean = {mean_use:.2f}")
    ax.axvline(median_use, color="#009E73", linestyle="--",
               label=f"median = {median_use:.1f}")
    ax.set_xlabel(f"times each {label} was sampled")
    ax.set_ylabel("donors")
    ax.set_yscale("log")
    ax.legend()
    ax.set_title(f"{len(usage):,} distinct donors used")
plt.tight_layout()
plt.show()

# Configuration

`base_presel` mirrors `simple_generic_histogram.ipynb`: drop the raw 1-gamma
overlay samples and require a reconstructed neutrino energy (the WC generic
selection).

In [ ]:
SEL_TITLE = "WC Generic Selection"

base_presel = all_df.filter(
    ~pl.col("filetype").is_in(["isotropic_one_gamma_overlay", "delete_one_gamma_overlay", "fullosc_overlay"])
    & (pl.col("wc_kine_reco_Enu") > 0)
)

# Open data comparisons

Nominal / `_deexadded` stacked-prediction pairs with Runs 1-5 open data
overlaid (`wc_net_weight_open_data`, prediction normalized per group to the
open-data POT as in `simple_generic_histogram.ipynb`). Statistical errors
only. Each pair shares one y-axis range so slides can be flipped back and
forth; the data points are identical in both plots of a pair (data events get
no injection), so flipping shows the MARLEY de-excitation prediction moving
under fixed data.

In [ ]:
weight_var_od = "wc_net_weight_open_data"


def plot_deex_pair_open_data(var, bins, display_var):
    """Nominal / _deexadded stacked-prediction pair with Runs 1-5 open data
    overlaid, sharing one y-axis range (both prediction variants and the data
    are included in the max)."""
    pred = base_presel.filter(pl.col(weight_var_od).is_not_null()
                              & (pl.col("filetype") != "data"))
    data = base_presel.filter(pl.col("filetype") == "data")
    pred_cols = pred.select([var, var + "_deexadded", weight_var_od]).collect()
    w = pred_cols[weight_var_od].to_numpy()
    data_vals = data.select(var).collect()[var].to_numpy()
    clip = lambda v: np.minimum(v, bins[-1] - 1e-9)
    max_total = max(
        max(np.histogram(clip(pred_cols[v].to_numpy()), bins=bins, weights=w)[0].max()
            for v in (var, var + "_deexadded")),
        np.histogram(clip(data_vals), bins=bins)[0].max(),
    )
    shared_ylim = (0, 1.2 * max_total)
    for suffix, tag in [("", "nominal"), ("_deexadded", "MARLEY de-excitation added")]:
        make_histogram_plot(
            pred_sel_df=pred, data_sel_df=data,
            bins=bins, var=var + suffix, display_var=display_var,
            net_weight_var=weight_var_od, data_type="Runs 1-5 Open Data",
            title=f"{SEL_TITLE} \u2014 Runs 1-5 Open Data \u2014 {tag}",
            ylim=shared_ylim,
        )

In [ ]:
plot_deex_pair_open_data("wc_blip_nWithin_75cm", np.arange(-0.5, 20.5, 1),
                        "Blips within 75 cm of WC reco $\\nu$ vertex")

In [ ]:
plot_deex_pair_open_data("wc_blip_minDist", np.linspace(0, 100, 26),
                        "Distance from WC reco $\\nu$ vertex to closest blip (cm)")

In [ ]:
plot_deex_pair_open_data("blip_sphere_n", np.arange(-0.5, 12.5, 1),
                        "Quality blips in 75 cm sphere around WC shower vertex")

In [ ]:
plot_deex_pair_open_data("blip_sphere_sumE", np.linspace(0, 25, 26),
                        "Summed quality-blip energy in 75 cm sphere (MeV)")

In [ ]:
plot_deex_pair_open_data("blip_no_shower_cone_n", np.arange(-0.5, 12.5, 1),
                        "Quality blips in sphere, outside shower cone")

In [ ]:
plot_deex_pair_open_data("blip_no_shower_cone_no_backtrack_cones_nonproton_n", np.arange(-0.5, 10.5, 1),
                        "Non-proton quality blips, no shower/backtrack cones")

# Erin Nn/0n proton multiplicity plots

Reproduces the final proton-count plots of `erin_blip_studies.ipynb` (Erin's
inclusive 1g selection and NC pi0 sideband, both with CRT veto, split into
Nn / 0n by the blip-based neutron cut), as nominal / `_deexadded` pairs with
open data. No systematics here (unlike the original notebook).

Both the plotted variable (reco protons + backtrack blips) and the Nn/0n split
itself are blip-based, so in the `_deexadded` version injected de-excitation
blips both migrate prediction events from 0n toward Nn and shift the proton+blip
count. Unlike the original notebook, run 4a is included here (matching the generic-selection
section's 9.57e19 POT); as in the original, overflow is not included (proton counts >= 4
are dropped).

In [ ]:
# mirrors erin_blip_studies.ipynb cell 3 (selections) and cell 5 (split/vars),
# except run 4a is kept here (the original notebook excluded it)
erin_presel = (
    all_df
    .filter(~pl.col("filetype").is_in([
        "isotropic_one_gamma_overlay", "delete_one_gamma_overlay",
        "numuCC_rad_corrected", "NC_coherent_1g_reweighted"
    ]))
    .filter(pl.col("wc_kine_reco_Enu") > 0)
    .with_columns(
        # _deexadded twin of the combined proton + backtrack-blip count
        # (built in postprocessing as blip_backtrack_cones_n + wc_reco_num_protons_35_MeV)
        (pl.col("blip_backtrack_cones_n_deexadded") + pl.col("wc_reco_num_protons_35_MeV"))
        .alias("wc_reco_num_protons_35_MeV_plus_backtrack_blips_deexadded")
    )
)

erin_1g_sel_expr = (
    (pl.col("wc_shw_sp_n_20mev_showers") > 0) &
    (pl.col("wc_reco_nuvtxX") > 5.0) & (pl.col("wc_reco_nuvtxX") < 250.0) &
    (pl.col("wc_single_photon_numu_score") > 0.4) &
    (pl.col("wc_single_photon_other_score") > 0.2) &
    (pl.col("wc_single_photon_ncpi0_score") > -0.05) &
    (pl.col("wc_single_photon_nue_score") > -1.0) &
    (pl.col("wc_shw_sp_n_20br1_showers") == 1)
)
erin_ncpi0_sel_expr = (
    (pl.col("wc_shw_sp_n_20mev_showers") > 0) &
    (pl.col("wc_reco_nuvtxX") > 5.0) & (pl.col("wc_reco_nuvtxX") < 250.0) &
    (pl.col("wc_single_photon_numu_score") > 0.1) &
    (pl.col("wc_single_photon_other_score") > -0.4) &
    (pl.col("wc_single_photon_ncpi0_score") < -0.4) &
    (pl.col("wc_single_photon_nue_score") > -20.0)
)

proton_bins = np.linspace(0, 4, 5)
oneshw_proton_blip_var = "wc_reco_num_protons_35_MeV_plus_backtrack_blips"
oneshwproton_blip_display_var = "WC Reconstructed num protons (35 MeV) + backtrack blips"


def plot_erin_proton_pair(sel_expr, split, title_base, breakdown):
    """Nominal / _deexadded pair of Erin-style proton+blip plots for one
    selection and one Nn/0n split, sharing a y-axis range. The Nn/0n split and
    the plotted variable both switch to their _deexadded twins in the second
    plot."""
    sel_df = erin_presel.filter(sel_expr & (pl.col("pandora_crtveto") == 0)).filter(
        pl.col("wc_net_weight_open_data").is_not_null() | (pl.col("filetype") == "data"))

    variants = []
    for suffix in ("", "_deexadded"):
        n_col = f"blip_no_shower_cone_no_backtrack_cones_n{suffix}"
        e_col = f"blip_no_shower_cone_no_backtrack_cones_sumE{suffix}"
        if split == "Nn":
            split_expr = (pl.col(n_col) > 2) | (pl.col(e_col) > 11)
        else:
            split_expr = (pl.col(n_col) <= 2) & (pl.col(e_col) <= 11)
        variants.append((suffix, split_expr, oneshw_proton_blip_var + suffix))

    ymax = 0.0
    for suffix, split_expr, var in variants:
        cols = sel_df.filter(split_expr).select([var, "wc_net_weight_open_data", "filetype"]).collect()
        is_data = (cols["filetype"] == "data").to_numpy()
        vals = cols[var].to_numpy()
        w = cols["wc_net_weight_open_data"].to_numpy()
        ymax = max(ymax,
                   np.histogram(vals[~is_data], bins=proton_bins, weights=w[~is_data])[0].max(),
                   np.histogram(vals[is_data], bins=proton_bins)[0].max())
    shared_ylim = (0, 1.3 * ymax)

    for (suffix, split_expr, var), tag in zip(variants, ("nominal", "MARLEY de-excitation added")):
        make_histogram_plot(
            pred_and_data_sel_df=sel_df.filter(split_expr),
            bins=proton_bins, var=var,
            display_var=oneshwproton_blip_display_var,
            net_weight_var="wc_net_weight_open_data", data_type="Runs 1-5 Open Data",
            title=f"{title_base}, {split} blip cut \u2014 {tag}",
            breakdown_type=breakdown,
            include_overflow=False, legend_fontsize=10, legend_ncol=1,
            ylim=shared_ylim,
        )

## Erin inclusive 1g selection (CRT veto)

In [ ]:
for split in ("Nn", "0n"):
    plot_erin_proton_pair(erin_1g_sel_expr, split,
                          "Erin Inclusive 1g Selection with CRT veto",
                          "erin_Np0pNn0n_categories")

## NC pi0 sideband (CRT veto)

In [ ]:
for split in ("Nn", "0n"):
    plot_erin_proton_pair(erin_ncpi0_sel_expr, split,
                          "Erin NC Pi0 Sideband with CRT veto",
                          "erin_Np0pNn0n_pi0_categories")

# Expectations

- Roughly 72% of matched overlay events receive >= 1 MARLEY photon (mean ~1.26),
  but after blip efficiency, TPC clipping, and out-of-TPC vertices, the mean
  injected reco blips per overlay event is ~0.4.
- `wc_blip_nWithin_75cm` should shift up by ~0.2 on average -- visible as a
  slight shift of the multiplicity distribution toward higher counts and a
  ratio above 1 that grows with blip count.
- The sphere/cone quality-cut variables shift less (their cuts absorb part of
  the injection), and proton-tagged variables should barely move (de-excitation
  blips are electron-like).
- This is the systematic variation itself: any selection variable that shifts
  appreciably here identifies where the de-excitation modeling uncertainty
  enters the analysis.